In [1]:
import tensorflow_hub as hub
import tensorflow_text as text
import tensorflow as tf

In [2]:
import pandas as pd

df = pd.read_csv("spam.csv")
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
df.groupby('Category').describe()

Message                                                            \
           count unique                                                top   
Category                                                                     
ham         4825   4516                             Sorry, I'll call later   
spam         747    641  Please call our customer service representativ...   

               
         freq  
Category       
ham        30  
spam        4

In [6]:
df['spam']=df['Category'].apply(lambda x: 1 if x=='spam' else 0)
df.head()

,Category,Message,spam
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [17]:
x = df['Message']
y = df['spam']

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [27]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(x)

In [28]:
import imblearn

In [30]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy='minority')
X_sm, y_sm = smote.fit_resample(X, y)

y_sm.value_counts()

spam
0    4825
1    4825
Name: count, dtype: int64

In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y, stratify=df['spam'])

In [39]:
X_train.shape, X_test.shape 

((4179, 8709), (1393, 8709))

In [42]:
y_train.shape, y_test.shape

((4179,), (1393,))

In [75]:
# Load the BERT preprocessing and encoder models
bert_preprocess = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3", name="bert_preprocess")
bert_encoder = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4", trainable=True, name="bert_encoder")

In [55]:
def get_sentence_embeding(sentences):
    preprocessed_text = bert_preprocess(sentences)
    return bert_encoder(preprocessed_text)['pooled_output']

get_sentence_embeding([
    "500$ discount. hurry up", 
    "Bhavin, are you up for a volleybal game tomorrow?"]
)

<tf.Tensor: shape=(2, 768), dtype=float32, numpy=
array([[-0.8435169 , -0.5132727 , -0.8884573 , ..., -0.7474887 ,
        -0.75314724,  0.91964495],
       [-0.87208366, -0.5054398 , -0.9444668 , ..., -0.8584751 ,
        -0.71745354,  0.88082975]], dtype=float32)>

In [56]:
e = get_sentence_embeding([
    "banana", 
    "grapes",
    "mango",
    "jeff bezos",
    "elon musk",
    "bill gates"
]
)

In [57]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_similarity([e[0]],[e[1]])

array([[0.9911088]], dtype=float32)

In [58]:
cosine_similarity([e[0]],[e[3]])


array([[0.84703827]], dtype=float32)

In [59]:
cosine_similarity([e[3]],[e[4]])

array([[0.98720354]], dtype=float32)

In [ ]:
import tensorflow as tf
from tensorflow import keras

model = tf.keras.Sequential([
    # Bert layers
    text_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='text')
    preprocessed_text = bert_preprocess(text_input)
    outputs = bert_encoder(preprocessed_text)

    
])

In [88]:
# Alternative approach - Sequential model with BERT

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(), dtype=tf.string, name='text'),
    hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3", 
                    name="bert_preprocess"),
    hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4", 
                    trainable=True, name="bert_encoder"),
    tf.keras.layers.Dropout(0.1, name="dropout"),
    tf.keras.layers.Dense(1, activation='sigmoid', name="output")
])



# Build alternative model
# model_alt = build_bert_model_sequential()
print("Alternative model built successfully!")


ValueError: Only instances of `keras.Layer` can be added to a Sequential model. Received: <tensorflow_hub.keras_layer.KerasLayer object at 0x13e0e6e90> (of type <class 'tensorflow_hub.keras_layer.KerasLayer'>)

In [86]:
model.summary()


NameError: name 'model' is not defined